# 1. Symmetric Encryption: One Key, Shared by Both Sides

Every authentication system eventually rests on a small set of cryptographic ideas.
This notebook covers the first and oldest one: **symmetric encryption**, where the
same key both locks and unlocks a message.

By the end of this notebook you should be able to answer:

- What does it mean for a cipher to be "symmetric"?
- What can symmetric encryption **not** solve on its own?
- Why does that gap motivate everything in the next notebook?

## 1.1 The idea, without any code

Imagine a locked box with exactly one key. Whoever holds a copy of that key can lock
the box and also unlock it. If Alice and Bob both have a copy of the same key, they
can exchange locked boxes freely, but so can anyone else who ever gets a copy of
that key.

That's the whole idea of symmetric encryption: **one shared secret, used for both
directions.**

## 1.2 A toy example: the Caesar cipher

Before reaching for a real cryptographic library, it's worth seeing the idea in its
simplest possible form: shifting letters by a fixed amount. This is not secure by
modern standards (it's crackable in seconds), but it shows the shape of the problem:
the "key" is just a number, and the same number encrypts and decrypts.

In [ ]:
def caesar_encrypt(text: str, shift: int) -> str:
    """Shift each letter in the text by a fixed amount."""
    result = []
    for ch in text:
        if ch.isalpha():  # Only shift letters, not spaces or punctuation
            base = ord('A') if ch.isupper() else ord('a')  # Preserve case (upper vs lower)
            # Shift the letter and wrap around using modulo 26 (number of letters in alphabet)
            result.append(chr((ord(ch) - base + shift) % 26 + base))
        else:
            result.append(ch)  # Keep non-letters unchanged
    return "".join(result)

def caesar_decrypt(text: str, shift: int) -> str:
    """Decrypt by shifting backwards (negative shift)."""
    return caesar_encrypt(text, -shift)

# The "key" is just a number: 7
secret_key = 7
message = "MEET ME AT THE LOBBY"

# Encrypt using the key
ciphertext = caesar_encrypt(message, secret_key)
# Decrypt using the same key
recovered = caesar_decrypt(ciphertext, secret_key)

print("Original: ", message)
print("Encrypted:", ciphertext)
print("Decrypted:", recovered)


Original:  MEET ME AT THE LOBBY
Encrypted: TLLA TL HA AOL SVIIF
Decrypted: MEET ME AT THE LOBBY


Notice: the number `7` played the role of key on **both** the way in and the way
out. That's the defining trait of a symmetric cipher, and it's true of real,
production-grade ciphers too, just with vastly more sophisticated math than
"shift each letter."

## 1.3 A real symmetric cipher: Fernet (AES under the hood)

Python's `cryptography` library provides `Fernet`, a symmetric encryption scheme
built on AES. It generates a proper random key, encrypts data with strong
authenticated encryption (meaning it also detects tampering), and is safe to use as
a building block in real systems, unlike our Caesar cipher toy above.

In [ ]:
from cryptography.fernet import Fernet

# Generate a random symmetric key. Whoever holds this key can encrypt AND decrypt.
# This is a real cryptographic key, much stronger than our "7" from before.
key = Fernet.generate_key()
print("Symmetric key (keep this secret!):", key)

# Create a cipher object that uses our key for encrypting/decrypting
cipher = Fernet(key)

# The message we want to protect
message = b"Meet me at the lobby at 9pm."
# Encrypt it using our key
token = cipher.encrypt(message)
print("\nCiphertext:", token)

# Decrypt it back to the original message using the same key
plaintext = cipher.decrypt(token)
print("\nDecrypted: ", plaintext.decode())


Symmetric key (keep this secret!): b'qsKlx-ANAhq5OOObQHJtLhsPL8c6L8sZuwSoWQr1SUE='

Ciphertext: b'gAAAAABqm0dlShumSYMZzfRTNABIMt9isZI1ACuN3_ct2a8ZkK2euLg8CQyXLCEYBrC7tFAcf72htipTnuxrL-a3kcl-sqBLF0VlE0iiczWr2rzvRYeseqI='

Decrypted:  Meet me at the lobby at 9pm.


Try it yourself: change even one byte of `token` before decrypting, and `Fernet`
will refuse to decrypt it rather than silently returning garbage. That "refuse to
decrypt tampered data" behavior (called **authenticated encryption**) is a preview
of a theme that comes back throughout this repository: a good cryptographic
primitive doesn't just hide data, it also proves the data wasn't altered.

In [ ]:
import binascii

# Create a mutable copy of the encrypted message so we can modify it
tampered = bytearray(token)
# Flip some bits in the middle of the ciphertext using XOR (^) with 0xFF
# This simulates what happens if an attacker intercepts and corrupts the message
tampered[10] ^= 0xFF
tampered = bytes(tampered)

# Try to decrypt the corrupted message
try:
    cipher.decrypt(tampered)
except Exception as e:
    # Fernet detects the tampering and refuses to decrypt
    print("Decryption correctly failed on tampered data:")
    print(" ", type(e).__name__, "-", e)


Decryption correctly failed on tampered data:
  InvalidToken - 


## 1.4 The problem symmetric encryption doesn't solve

Symmetric encryption works great **once both sides already share a secret key**.
But that raises an obvious question:

> How did Alice and Bob agree on that key in the first place, over a network that
> might be watched by an attacker?

If Alice just emails Bob the key, anyone reading their email now has it too. This is
called the **key distribution problem**, and it's a real, practical limitation,
not a theoretical nitpick. Every time you log into a website, you and that website
have never met before, and you have no pre-shared secret with it.

This is exactly the gap that **asymmetric (public/private key) cryptography**
solves, which is the subject of the next notebook.

## Summary

- Symmetric encryption uses **one key** for both encryption and decryption.
- It's fast and effective, but requires both parties to already share that key
  secretly.
- Modern authenticated symmetric encryption (like `Fernet`/AES) also detects
  tampering, not just hides content.
- The unsolved problem: **how do two strangers agree on a secret over an open
  network?** This motivates asymmetric cryptography, next.

**Next:** `02_asymmetric_keypairs.ipynb`

## Appendix: Modules and Functions Used

### cryptography.fernet (The Fernet module)
A Python library for symmetric encryption. Think of it as a secure padlock that uses a single key to lock and unlock data. Unlike our toy Caesar cipher, Fernet uses industrial-strength cryptography (AES algorithm) and automatically detects if someone tried to tamper with your encrypted message. It's safe to use in real applications.

**Functions used:**

- `Fernet.generate_key()`: Creates a new random secret key. This key is what locks and unlocks your messages. Keep it secret!
  
- `Fernet(key)`: Creates a cipher object that knows how to encrypt and decrypt using your key. Think of this as "loading the key into the lock."
  
- `.encrypt(message)`: Scrambles your message using the key so only someone with the key can read it. Returns the scrambled (encrypted) version.
  
- `.decrypt(encrypted_message)`: Unscrambles an encrypted message back to readable form if you have the correct key. Will refuse to decrypt if the message was tampered with.